# QuantAlpha AI — Step 6: Extended History (10 Years) + Multi-Cycle Signal Re-Test

**Why this matters:** everything tested so far (Steps 3B, 4C) used only 3 years of data — which
very likely covers mostly ONE bull market. A signal that looks fine in a bull run can completely
fail in a bear or sideways market. This notebook:

1. Re-fetches 10 years of OHLCV for all Nifty 200 stocks (separate from your original 3-year data
   — saved to a new folder, doesn't overwrite Step 1's work)
2. Recomputes technical indicators on the longer history
3. Re-runs the SAME signal tests from Step 3B across this longer window, split into distinct
   sub-periods (so we can see if a signal's edge holds up, flips, or disappears across different
   market conditions — e.g., 2016-2020 vs 2020-2022 vs 2022-2026)
4. Reports results honestly per sub-period, not just one blended number


## 1. Connect to Drive

In [5]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/QuantAlpha')
os.makedirs('data_10yr/ohlcv', exist_ok=True)
os.makedirs('data_10yr/technical', exist_ok=True)
print("Ready.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ready.


In [6]:
!pip install yfinance pandas-ta
print("Dependencies ready.")


Dependencies ready.


In [7]:
import pandas as pd
import numpy as np
import yfinance as yf
import pandas_ta as ta
import time
import glob
import warnings
warnings.filterwarnings('ignore')


## 2. Re-fetch 10-year OHLCV for all Nifty 200 stocks
This takes longer than the original 3-year fetch (more data per stock). Expect 25-40 minutes.


In [9]:
technical_files_3yr = glob.glob('data/technical/*.parquet')
symbols = sorted(f.split('/')[-1].replace('.parquet', '') for f in technical_files_3yr)
print(f"Re-fetching 10-year history for {len(symbols)} stocks (reusing your existing Nifty 200 list)")

def fetch_ohlcv_10yr(symbol, retries=3):
    for attempt in range(retries):
        try:
            df = yf.Ticker(symbol + ".NS").history(period="10y", interval="1d")
            if df.empty:
                return None
            df = df.reset_index()
            return df
        except Exception:
            time.sleep(2)
    return None

fetched = 0
failed = []
for i, sym in enumerate(symbols):
    df = fetch_ohlcv_10yr(sym)
    if df is not None and len(df) > 500:
        df.to_parquet(f"data_10yr/ohlcv/{sym}.parquet")
        fetched += 1
    else:
        failed.append(sym)
    if i % 20 == 0:
        print(f"Progress: {i}/{len(symbols)} | OK: {fetched} | Failed: {len(failed)}")
    time.sleep(0.3)

print(f"\nDone. Fetched 10-year history for {fetched} stocks. Failed: {len(failed)}")
print("Note: some stocks (recent IPOs) genuinely won't have 10 years of history -- that's expected, not an error.")


Re-fetching 10-year history for 200 stocks (reusing your existing Nifty 200 list)
Progress: 0/200 | OK: 1 | Failed: 0
Progress: 20/200 | OK: 21 | Failed: 0
Progress: 40/200 | OK: 41 | Failed: 0
Progress: 60/200 | OK: 60 | Failed: 1
Progress: 80/200 | OK: 78 | Failed: 3
Progress: 100/200 | OK: 97 | Failed: 4
Progress: 120/200 | OK: 115 | Failed: 6
Progress: 140/200 | OK: 135 | Failed: 6
Progress: 160/200 | OK: 154 | Failed: 7
Progress: 180/200 | OK: 172 | Failed: 9

Done. Fetched 10-year history for 188 stocks. Failed: 12
Note: some stocks (recent IPOs) genuinely won't have 10 years of history -- that's expected, not an error.


## 3. Compute technical indicators on the extended history

In [10]:
def compute_technical_indicators(df):
    df = df.copy()
    df['RSI_14'] = ta.rsi(df['Close'], length=14)
    df['ATR_14'] = ta.atr(df['High'], df['Low'], df['Close'], length=14)
    df['SMA_50'] = ta.sma(df['Close'], length=50)
    df['SMA_200'] = ta.sma(df['Close'], length=200)
    adx_result = ta.adx(df['High'], df['Low'], df['Close'], length=14)
    df['ADX_14'] = adx_result['ADX_14'] if adx_result is not None else np.nan
    return df

ohlcv_10yr_files = glob.glob('data_10yr/ohlcv/*.parquet')
symbols_10yr = sorted(f.split('/')[-1].replace('.parquet', '') for f in ohlcv_10yr_files)

for sym in symbols_10yr:
    df = pd.read_parquet(f"data_10yr/ohlcv/{sym}.parquet")
    enriched = compute_technical_indicators(df)
    enriched.to_parquet(f"data_10yr/technical/{sym}.parquet")

print(f"Computed indicators for {len(symbols_10yr)} stocks with 10-year history")


Computed indicators for 188 stocks with 10-year history


## 4. Define market sub-periods to test separately
Rough, well-known Indian market regime windows (approximate, for directional testing):
- 2016-2019: broadly bullish, pre-COVID
- 2020-early 2020: COVID crash (sharp bear/volatile)
- 2020-2021: sharp recovery/bull run
- 2022: sideways/correction (rate hikes, FII outflows)
- 2023-2026: mixed/recent


In [11]:
def get_date_ranges(df):
    df['Date'] = pd.to_datetime(df['Date']).dt.tz_localize(None)
    return df

def label_period(date):
    if date < pd.Timestamp('2020-01-01'):
        return '2016-2019 (pre-COVID bull)'
    elif date < pd.Timestamp('2020-07-01'):
        return '2020 H1 (COVID crash)'
    elif date < pd.Timestamp('2022-01-01'):
        return '2020-2021 (recovery bull)'
    elif date < pd.Timestamp('2023-01-01'):
        return '2022 (sideways/correction)'
    else:
        return '2023-2026 (recent)'

print("Period labels defined.")


Period labels defined.


## 5. Re-run the trend-confirmed signal test (best performer from Step 3B) across each period
Testing the SAME signal logic from Step 3B (`trend_confirmed`: price above 50 & 200-day SMA, RSI
40-70) but now broken out by market regime, to see if its edge (or lack of it) is consistent or
regime-dependent.


In [12]:
HOLDING_DAYS = 15

all_signal_results = []

for sym in symbols_10yr:
    df = pd.read_parquet(f"data_10yr/technical/{sym}.parquet")
    df = get_date_ranges(df).reset_index(drop=True)
    if len(df) < 300:
        continue

    for i in range(200, len(df) - HOLDING_DAYS):
        close = df.loc[i, 'Close']
        sma50 = df.loc[i, 'SMA_50']
        sma200 = df.loc[i, 'SMA_200']
        rsi = df.loc[i, 'RSI_14']
        if pd.isna(sma50) or pd.isna(sma200) or pd.isna(rsi):
            continue

        signal_fired = (close > sma50) and (close > sma200) and (40 <= rsi <= 70)
        exit_price = df.loc[i + HOLDING_DAYS, 'Close']
        fwd_return = (exit_price - close) / close
        period_label = label_period(df.loc[i, 'Date'])

        all_signal_results.append({
            'symbol': sym,
            'period': period_label,
            'signal_fired': signal_fired,
            'fwd_return': fwd_return
        })

signal_df = pd.DataFrame(all_signal_results)
print(f"Total observations across all periods: {len(signal_df)}")


Total observations across all periods: 397238


## 6. Results — win rate and edge, broken out BY MARKET PERIOD

In [13]:
period_order = ['2016-2019 (pre-COVID bull)', '2020 H1 (COVID crash)',
                '2020-2021 (recovery bull)', '2022 (sideways/correction)', '2023-2026 (recent)']

results = []
for period in period_order:
    subset = signal_df[signal_df['period'] == period]
    if len(subset) == 0:
        continue
    fired = subset[subset['signal_fired'] == True]
    not_fired = subset[subset['signal_fired'] == False]
    if len(fired) < 30:
        continue
    win_rate_fired = (fired['fwd_return'] > 0).mean() * 100
    win_rate_baseline = (not_fired['fwd_return'] > 0).mean() * 100
    avg_return_fired = fired['fwd_return'].mean() * 100

    results.append({
        'period': period,
        'n_signal_obs': len(fired),
        'win_rate_when_fired': round(win_rate_fired, 1),
        'win_rate_baseline': round(win_rate_baseline, 1),
        'edge_pct_points': round(win_rate_fired - win_rate_baseline, 1),
        'avg_return_pct': round(avg_return_fired, 2)
    })

results_summary = pd.DataFrame(results)
print("=== 'trend_confirmed' signal — edge broken out by market regime ===\n")
print(results_summary.to_string(index=False))
results_summary.to_csv('data_10yr/signal_by_market_regime.csv', index=False)


=== 'trend_confirmed' signal — edge broken out by market regime ===

                    period  n_signal_obs  win_rate_when_fired  win_rate_baseline  edge_pct_points  avg_return_pct
2016-2019 (pre-COVID bull)         38744                 51.3               51.4             -0.1            0.52
     2020 H1 (COVID crash)          4801                 47.1               53.9             -6.7           -1.78
 2020-2021 (recovery bull)         33708                 59.8               61.1             -1.3            3.28
2022 (sideways/correction)         15936                 48.3               54.1             -5.8            0.30
        2023-2026 (recent)         66911                 57.2               55.5              1.7            1.77


## 7. How to read this honestly

- **If the edge is positive and similar across ALL periods** (bull, bear, sideways) — that's a
  genuinely robust signal, much more trustworthy than the 3-year-only test
- **If the edge is only positive in bull periods (2016-2019, 2020-2021) and flips negative in
  2020 H1 or 2022** — that confirms the signal only works in favorable conditions, which is a
  real, important limitation to document (and matches what we suspected from the 3-year test)
- **If edge is small/inconsistent everywhere** — reinforces the Step 3B finding: simple technical
  signals don't have robust standalone edge on this universe, regardless of market regime

This result should replace any single-window claim — always describe your signal's performance
per regime, not as one blended number, in your documentation.


## 8. Summary
Saved: `data_10yr/signal_by_market_regime.csv`

This is now your most defensible technical-signal test — spanning bull, bear (COVID), sideways
(2022), and recent periods, not just one 3-year bull run. Whatever the result, it's honest and
presentable in an interview: *"I tested across multiple market regimes, not just one favorable
window, and here's exactly what held up and what didn't."*

**Next: GitHub + README** — time to document everything, including this multi-regime honesty,
as your portfolio narrative.
